# Requirements

This notebook assumes the Oracle database already contains the ingested `research_papers` table, embeddings, and related search indexes.

In [43]:
import oracledb
import time

def connect_to_oracle(max_retries=3, retry_delay=5):
    """
    Connect to Oracle database with retry logic and better error handling.
    
    Args:
        max_retries: Maximum number of connection attempts
        retry_delay: Seconds to wait between retries
    """
    user = "system"
    password = "OraclePwd_2025"  # must match ORACLE_PWD from docker run
    dsn = "localhost:1521/FREEPDB1"
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(
                user=user,
                password=password,
                dsn=dsn
            )
            print("✓ Connected successfully!")
            
            # Test the connection
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%';")
                banner = cur.fetchone()[0]
                print(f"\n{banner}")
            
            return conn
            
        except oracledb.OperationalError as e:
            error_msg = str(e)
            print(f"✗ Connection failed (attempt {attempt}/{max_retries})")
            
            if "DPY-4011" in error_msg or "Connection reset by peer" in error_msg:
                print("  → This usually means:")
                print("    1. Database is still starting up (wait 2-3 minutes)")
                print("    2. Listener is not bound to 0.0.0.0 (run fix_oracle_listener())")
                print("    3. Container is not running (check with check_docker_container())")
                
                if attempt < max_retries:
                    print(f"\n  Waiting {retry_delay} seconds before retry...")
                    time.sleep(retry_delay)
                else:
                    print("\n  💡 Try running:")
                    print("     1. check_docker_container() - verify container is running")
                    print("     2. fix_oracle_listener() - fix listener binding")
                    raise
            else:
                raise
        except Exception as e:
            print(f"✗ Unexpected error: {e}")
            raise
    
    raise ConnectionError("Failed to connect after all retries")

# Connect to Oracle
conn = connect_to_oracle()

Connection attempt 1/3...
✓ Connected successfully!

Oracle AI Database 26ai Free Release 23.26.1.0.0 - Develop, Learn, and Run for Free


In [20]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("nomic-ai/nomic-embed-text-v1.5", trust_remote_code=True)

<All keys matched successfully>


In [21]:
import numpy as np
import array

In [22]:
def keyword_search_research_papers(conn, keyword: str):
    """
    Perform a full-text keyword search on the 'text' column 
    using the Oracle Text index (rp_text_idx).

    Args:
        conn: Oracle database connection object.
        keyword (str): Keyword or phrase to search for.

    Returns:
        tuple: (rows, columns)
    """
    query = """
        SELECT 
            arxiv_id, 
            title, 
            SUBSTR(text, 1, 200) AS text_snippet,
            SCORE(1) AS relevance_score
        FROM research_papers
        WHERE CONTAINS(text, :keyword, 1) > 0
        ORDER BY SCORE(1) DESC
        FETCH FIRST 10 ROWS ONLY
    """

    with conn.cursor() as cur:
        cur.execute(query, keyword=keyword)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    return rows, columns


In [23]:
SEARCH_QUERY = "Get me papers related to planetary exploration"

In [24]:
def vector_search_research_papers(conn, embedding_model, search_query: str, top_k: int = 5):
    """
    Perform a vector similarity search on the research_papers table using a query embedding.
    Returns cosine similarity scores (higher = more similar).
    """

    # 1️⃣ Encode the query into a vector
    query_embedding = embedding_model.encode(
        [f"search_query: {search_query}"], 
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32).tolist()

    # 2️⃣ Prepare the vector for Oracle binding
    query_embedding_array = array.array('f', query_embedding)

    # 3️⃣ Run a vector similarity search using cosine similarity
    query = f"""
        SELECT 
            arxiv_id, 
            title, 
            abstract, 
            SUBSTR(text, 1, 200) AS text_snippet,
            ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
        FROM research_papers
        ORDER BY similarity_score DESC
        FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
    """

    # 4️⃣ Execute and return results
    with conn.cursor() as cur:
        cur.execute(query, q=query_embedding_array)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    return rows, columns


In [25]:
import array
import numpy as np

def hybrid_search_research_papers_pre_filter(
    conn,
    embedding_model,
    search_phrase: str,
    top_k: int = 10,
    show_explain: bool = False
):
    """
    Perform a hybrid search using Oracle Text + Vector Search.
    Combines lexical filtering (CONTAINS) with semantic re-ranking via cosine similarity.

    Args:
        conn: Oracle database connection object.
        embedding_model: Model with `.encode()` method (e.g., SentenceTransformer).
        search_phrase (str): User search phrase used for both text filtering and embedding.
        top_k (int): Number of results to return (default = 10).
        show_explain (bool): If True, prints the execution plan.

    Returns:
        tuple: (rows, columns, exec_plan_text or None)
    """

    # --- Step 1: Encode search phrase into normalized vector ---
    query_embedding = embedding_model.encode(
        [f"search_query: {search_phrase}"],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0].astype(np.float32).tolist()
    query_embedding_array = array.array('f', query_embedding)

    with conn.cursor() as cur:
        # Enable runtime stats if needed
        if show_explain:
            cur.execute("ALTER SESSION SET statistics_level = ALL")

        # --- Step 2: Hybrid query (Oracle Text + Vector) ---
        sql = f"""
            SELECT {"/*+ GATHER_PLAN_STATISTICS */" if show_explain else ""}
                arxiv_id,
                title,
                abstract,
                SUBSTR(text, 1, 200) AS text_snippet,
                ROUND(1 - VECTOR_DISTANCE(embedding, :q, COSINE), 4) AS similarity_score
            FROM research_papers
            WHERE CONTAINS(text, :kw, 1) > 0
            ORDER BY similarity_score DESC
            FETCH APPROX FIRST {top_k} ROWS ONLY WITH TARGET ACCURACY 90
        """

        cur.execute(sql, q=query_embedding_array, kw=search_phrase)
        rows = cur.fetchall()
        columns = [desc[0] for desc in cur.description]

    # --- Step 3: Execution plan (optional) ---
    exec_plan_text = None
    if show_explain:
        with conn.cursor() as cur_plan:
            cur_plan.execute("""
                SELECT plan_table_output
                FROM TABLE(DBMS_XPLAN.DISPLAY_CURSOR(NULL, NULL, 'ALLSTATS LAST +PREDICATE'))
            """)
            exec_plan_text = "\n".join(r[0] for r in cur_plan.fetchall())

        print("\n====== Execution Plan (DBMS_XPLAN.DISPLAY_CURSOR) ======")
        print(exec_plan_text)
        print("========================================================\n")

    return rows, columns, exec_plan_text


In [ ]:
# import getpass
# import os

# # Function to securely get and set environment variables
# def set_env_securely(var_name, prompt):
#     value = getpass.getpass(prompt)
#     os.environ[var_name] = value

In [27]:
# Azure OpenAI environment setup helpers
import getpass
import os

# Function to securely get and set environment variables
def set_env_securely_azure(var_name, prompt):
    value = getpass.getpass(prompt)
    os.environ[var_name] = value

In [ ]:
# set_env_securely("OPENAI_API_KEY", "Enter your OPEN API Key: ")

In [29]:
# https://azure-agent-ai-foundry-resource.openai.azure.com/
# gpt-4o
# gpt-4.1
# gpt-5

set_env_securely_azure("AZURE_OPENAI_ENDPOINT", "Enter your Azure OpenAI endpoint (e.g. https://<resource>.openai.azure.com): ")
set_env_securely_azure("AZURE_OPENAI_DEPLOYMENT", "Enter your Azure OpenAI deployment name (e.g. gpt-4o): ")
print("RBAC auth enabled: ensure you are signed in (for example, via 'az login') and have Azure OpenAI permissions.")

RBAC auth enabled: ensure you are signed in (for example, via 'az login') and have Azure OpenAI permissions.


In [46]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM RESEARCH_PAPERS")
    print("Row count:", cur.fetchone()[0])

    cur.execute("""
        SELECT arxiv_id, title, abstract, text FROM RESEARCH_PAPERS
        FETCH FIRST 3 ROWS ONLY
    """)
    for row in cur.fetchall():
        print(row)

Row count: 1000
('0902.0428', 'Dynamics of planets in retrograde mean motion resonance', 'In a previous paper (Gayon &amp; Bois 2008a), we have shown the general efficiency of retrograde resonances for stabilizing compact planetary systems. Such retrograde resonances can be found when two-planets of a three-body planetary system are both in mean motion resonance and revolve in opposite directions. For a particular two-planet system, we have also obtained a new orbital fit involving such a counter-revolving configuration and consistent with the observational data. <br>In the present paper, we analytically investigate the three-body problem in this particular case of retrograde resonances. We therefore define a new set of canonical variables allowing to express correctly the resonance angles and obtain the Hamiltonian of a system harboring planets revolving in opposite directions. The acquiring of an analytical &#34;rail&#34; may notably contribute to a deeper understanding of our numeri

# Part 6: AI Agents with OpenAI and Oracle AI Database


In [ ]:
# OPENAI_MODEL = "gpt-4"

In [47]:
# Azure companion: use deployment name (not base model name)
AZURE_OPENAI_MODEL = "gpt-4.1"  # replace with your Azure OpenAI deployment name

In [48]:
from agents import Agent, Runner

In [ ]:
# research_paper_assistant = Agent(
#     name="Research Paper Assistant",
#     model=OPENAI_MODEL,
#     instructions="""
#       You are a Research Paper Assistant focused on helping users explore, analyze, and summarize
#       academic research.

#       Maintain a professional, concise, and scholarly tone appropriate for research discussions.
#     """,
# )


In [49]:
# Azure companion: configure openai-agents to use Azure OpenAI via DefaultAzureCredential
import os
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AsyncAzureOpenAI
from agents import Agent, set_default_openai_client, set_tracing_disabled

credential = DefaultAzureCredential()
token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default",
)

# Normalize endpoint in case env var includes /openai or deployment path.
raw_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
if "/openai" in raw_endpoint.lower():
    raw_endpoint = raw_endpoint[: raw_endpoint.lower().index("/openai")]

AZURE_OPENAI_MODEL = os.environ["AZURE_OPENAI_DEPLOYMENT"]
# Runner uses Responses API; 2024-10-21 commonly fails on /responses in Azure.
env_api_version = os.environ.get("AZURE_OPENAI_API_VERSION")
if not env_api_version or env_api_version == "2024-10-21":
    AZURE_OPENAI_API_VERSION = "2025-03-01-preview"
else:
    AZURE_OPENAI_API_VERSION = env_api_version

azure_agents_client = AsyncAzureOpenAI(
    azure_endpoint=raw_endpoint,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_ad_token_provider=token_provider,
    # Compatibility for openai<2.x credential gate when using only AAD token provider.
    _enforce_credentials=False,
)

# Guard: fail fast if this ever gets replaced with a sync client
if not isinstance(azure_agents_client, AsyncAzureOpenAI):
    raise TypeError(
        "Expected AsyncAzureOpenAI for agent runs. Restart kernel and rerun this cell before Azure runs."
    )

print(type(azure_agents_client))
print(f"Azure endpoint: {raw_endpoint}")
print(f"Azure deployment: {AZURE_OPENAI_MODEL}")
print(f"Azure API version: {AZURE_OPENAI_API_VERSION}")

# Route Agent/Runner calls to Azure OpenAI for the Azure companion cells
set_default_openai_client(azure_agents_client)
set_tracing_disabled(disabled=True)

research_paper_assistant_azure = Agent(
    name="Research Paper Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="""
      You are a Research Paper Assistant focused on helping users explore, analyze, and summarize
      academic research.

      Maintain a professional, concise, and scholarly tone appropriate for research discussions.
    """,
)

<class 'openai.lib.azure.AsyncAzureOpenAI'>
Azure endpoint: https://azure-agent-ai-foundry-resource.openai.azure.com
Azure deployment: gpt-4o
Azure API version: 2025-03-01-preview


In [ ]:
# # Optional helper: switch the agents runtime back to standard OpenAI
# from openai import AsyncOpenAI

# def switch_agents_runtime_to_openai():
#     if not os.environ.get("OPENAI_API_KEY"):
#         raise ValueError("OPENAI_API_KEY is required to switch back to standard OpenAI.")
#     set_default_openai_client(AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"]))
#     set_tracing_disabled(disabled=True)
#     print("Switched agents runtime back to standard OpenAI client.")

In [ ]:
# run_result = await Runner.run(
#     starting_agent=research_paper_assistant,
#     input="Summarize recent research on optimization techniques for planetary exploration.",
# )

In [50]:
import inspect
from openai import NotFoundError

# Clean up stale coroutine from previous failed runs to avoid noisy RuntimeWarning.
if "run_result_azure" in globals() and inspect.iscoroutine(run_result_azure):
    run_result_azure.close()

try:
    run_result_azure = await Runner.run(
        starting_agent=research_paper_assistant_azure,
        input="Summarize recent research on optimization techniques for planetary exploration.",
    )
except NotFoundError as e:
    raise RuntimeError(
        "Azure returned 404 (Resource not found). Verify AZURE_OPENAI_ENDPOINT points to the resource root "
        "(for example, https://<resource>.openai.azure.com) and AZURE_OPENAI_DEPLOYMENT matches an existing "
        "chat-capable deployment name. Then rerun Cell 99 and this cell."
    ) from e

In [ ]:
# print(run_result.final_output)


In [51]:
print(run_result_azure.final_output)


Recent research on optimization techniques for planetary exploration has primarily focused on enhancing resource efficiency, mission planning, and autonomous decision-making to navigate challenging extraterrestrial environments. Key advancements can be summarized across several areas:

### 1. **Trajectory Optimization**
Recent studies have examined efficient trajectory planning for spacecraft using reinforcement learning, heuristic algorithms (e.g., Genetic Algorithms, Particle Swarm Optimization), and machine learning-based predictive models. These methods aim to minimize propellant use, time, and energy while accounting for gravitational influences and planetary dynamics.

### 2. **Rover Path Planning**
Optimization techniques for planetary surface exploration rovers have increasingly applied D* Lite, A* algorithms, Artificial Neural Networks (ANNs), and Deep Reinforcement Learning (DRL) to identify safe, energy-efficient paths in uneven terrain. Enhanced perception systems include t

In [52]:
from agents.tool import function_tool

@function_tool
def get_research_papers(user_query: str, retrieval_mode: str = "hybrid", top_k: int = 5) -> str:
    """
    Retrieves academic research papers relevant to the user's query.

    This tool queries the research_papers SQL table using one of three retrieval techniques:
        - 'keyword'  → lexical search via LIKE filtering
        - 'vector'   → semantic similarity search
        - 'hybrid'   → combines keyword prefiltering + vector similarity (default)

    Use this tool when analyzing or summarizing scientific literature.

    Args:
        user_query (str): Research topic or question to search for.
        retrieval_mode (str): 'keyword', 'vector', or 'hybrid'. Default is 'hybrid'.
        top_k (int): Number of top papers to retrieve (default=5).

    Returns:
        str: A formatted summary of the most relevant research papers.
    """

    # ------------------------------------------------------------------
    # Perform retrieval using SQL-based functions (defined earlier)
    # ------------------------------------------------------------------
    if retrieval_mode == "keyword":
        rows, columns = keyword_search_research_papers(conn, user_query)
    elif retrieval_mode == "vector":
        rows, columns = vector_search_research_papers(conn, embedding_model, user_query, top_k)
    else:
        rows, columns, _ = hybrid_search_research_papers_pre_filter(
            conn=conn,
            embedding_model=embedding_model,
            search_phrase=user_query,
            top_k=top_k,
            show_explain=False
        )

    retrieved_count = len(rows) if rows else 0

    # ------------------------------------------------------------------
    # Format the output into a readable string
    # ------------------------------------------------------------------
    if retrieved_count == 0:
        return f"No research papers found related to '{user_query}'."

    formatted_results = [f"📚 {retrieved_count} papers retrieved for query: '{user_query}'\n"]
    for i, row in enumerate(rows):
        row_data = dict(zip(columns, row))
        title = row_data.get("TITLE", "Untitled Paper")
        abstract = row_data.get("ABSTRACT", "No abstract available.")
        score = (
            row_data.get("SIMILARITY_SCORE")
            or row_data.get("TEXT_RELEVANCE_SCORE")
            or "N/A"
        )
        formatted_results.append(
            f"[{i+1}] {title}\n"
            f"Abstract: {abstract}\n"
            f"Relevance Score: {score}\n"
        )

    return "\n".join(formatted_results)


In [ ]:
# research_paper_assistant.tools.append(get_research_papers)

In [53]:
research_paper_assistant_azure.tools.append(get_research_papers)

In [ ]:
# run_result_with_tool = await Runner.run(
#     starting_agent=research_paper_assistant,
#     input="Get me information on rover navigation, planetary data collection, mission planning, resource allocation, or other related fields",
# )

In [54]:
run_result_with_tool_azure = await Runner.run(
    starting_agent=research_paper_assistant_azure,
    input="Get me information on rover navigation, planetary data collection, mission planning, resource allocation, or other related fields",
)

In [ ]:
# print(run_result_with_tool.final_output)

In [55]:
print(run_result_with_tool_azure.final_output)

One relevant paper retrieved focuses on mission planning and data collection related to the EPOXI mission targeting comet 103P/Hartley 2. Below are the key details:

**Title:** "The nucleus of 103P/Hartley 2, target of the EPOXI mission"

**Abstract Highlights:**
- The study involves observations of comet 103P/Hartley 2, selected for the Deep Impact extended EPOXI mission.
- Objectives included verifying the comet’s position/activity, providing astrometry and brightness data, and aiding mission planning.
- Observations were conducted at heliocentric distances between 5.7 and 5.5 AU using the VLT's FORS2 at different epochs.
- Photometry revealed a low nucleus radius (≤1 km) and faint activity continuing past aphelion, indicative of prior appearances rather than new perihelion activity.

This paper combines aspects of astrophysical analysis, resource allocation for optical observation, and mission navigation planning, particularly for cometary studies during planetary missions. 

Let me

In [ ]:
# import pprint
# pprint.pprint(run_result_with_tool.raw_responses)

In [56]:
import pprint
pprint.pprint(run_result_with_tool_azure.raw_responses)

[ModelResponse(output=[ResponseFunctionToolCall(arguments='{"user_query":"rover navigation, planetary data collection, mission planning, resource allocation","retrieval_mode":"hybrid","top_k":5}', call_id='call_LQiIzvKUm9fSxF8Nl8taB3Zc', name='get_research_papers', type='function_call', id='fc_003a03a7df6cb5dd006a1729d4f3e88193af32eb84265bd5ff', namespace=None, status='completed')],
               usage=Usage(requests=1,
                           input_tokens=190,
                           input_tokens_details=InputTokensDetails(cached_tokens=0),
                           output_tokens=43,
                           output_tokens_details=OutputTokensDetails(reasoning_tokens=0),
                           total_tokens=233,
                           request_usage_entries=[]),
               response_id='resp_003a03a7df6cb5dd006a1729d416f4819387e11a11996ba9f2',
               request_id='d0530094-0a36-44a6-8708-85ca231f0e77'),
 ModelResponse(output=[ResponseOutputMessage(id='msg_003a0

### Build an Agent with Multiple Tool Access

In [57]:
from agents.tool import function_tool

@function_tool
def get_past_research_conversations(user_query: str, top_k: int = 5) -> str:
    """
    Retrieves relevant past research-related conversations or analyses related to the query.

    This tool searches a SQL database of prior research assistant conversations, 
    literature discussions, or synthesis sessions to find relevant context. 
    It allows the research assistant to recall previous analyses or summaries 
    that addressed similar topics, providing continuity and richer insights.

    Args:
        user_query (str): The research topic, concept, or question to search for.
        top_k (int): Number of top past discussions to retrieve (default=5).

    Returns:
        str: Formatted examples of relevant past research discussions.
    """

    # ------------------------------------------------------------------
    # Perform retrieval using the SQL-based hybrid search (vector + keyword)
    # ------------------------------------------------------------------
    rows, columns, _ = hybrid_search_research_papers_pre_filter(
        conn=conn,
        embedding_model=embedding_model,
        search_phrase=user_query,
        top_k=top_k,
        show_explain=False
    )

    retrieved_count = len(rows) if rows else 0

    # ------------------------------------------------------------------
    # Format results for readability
    # ------------------------------------------------------------------
    if retrieved_count == 0:
        return f"No past research discussions found related to '{user_query}'."

    formatted_results = [f"🧠 {retrieved_count} past research discussions retrieved for query: '{user_query}'\n"]
    for i, row in enumerate(rows):
        row_data = dict(zip(columns, row))
        title = row_data.get("TITLE", "Untitled Discussion")
        abstract = row_data.get("ABSTRACT", "No summary available.")
        snippet = row_data.get("TEXT_SNIPPET", "")
        score = (
            row_data.get("SIMILARITY_SCORE")
            or row_data.get("TEXT_RELEVANCE_SCORE")
            or "N/A"
        )
        formatted_results.append(
            f"[{i+1}] **{title}**\n"
            f"Summary: {abstract}\n"
            f"Snippet: {snippet}\n"
            f"Relevance Score: {score}\n"
        )

    return "\n".join(formatted_results)


Let's update our agent instruction to ensure it knows when to utilize the right tools

In [ ]:
# upgraded_research_paper_assistant = Agent(
#     name="Research Paper Assistant",
#     model=OPENAI_MODEL,
#     instructions="""
#     Always maintain an academic, evidence-based tone.
#     Your purpose is to help users explore, synthesize, and connect research insights —
#     not to speculate or fabricate information.
#     """,
# )


In [58]:
upgraded_research_paper_assistant_azure = Agent(
    name="Research Paper Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="""
    Always maintain an academic, evidence-based tone.
    Your purpose is to help users explore, synthesize, and connect research insights —
    not to speculate or fabricate information.
    """,
)

In [ ]:
# # Attach research retrieval tools to the upgraded research assistant
# upgraded_research_paper_assistant.tools.append(get_research_papers)
# upgraded_research_paper_assistant.tools.append(get_past_research_conversations)

In [59]:
# Attach research retrieval tools to the upgraded Azure research assistant
upgraded_research_paper_assistant_azure.tools.append(get_research_papers)
upgraded_research_paper_assistant_azure.tools.append(get_past_research_conversations)

In [ ]:
# pprint.pprint(upgraded_research_paper_assistant.tools)

In [60]:
pprint.pprint(upgraded_research_paper_assistant_azure.tools)

[FunctionTool(name='get_research_papers',
              description='Retrieves academic research papers relevant to the '
                          "user's query.",
              params_json_schema={'additionalProperties': False,
                                  'properties': {'retrieval_mode': {'default': 'hybrid',
                                                                    'description': "'keyword', "
                                                                                   "'vector', "
                                                                                   'or '
                                                                                   "'hybrid'. "
                                                                                   'Default '
                                                                                   'is '
                                                                                   "'hybrid'.",
                        

In [ ]:
# run_result_with_tools = await Runner.run(
#     starting_agent=upgraded_research_paper_assistant,
#     input=(
#         "Get me information on rover navigation, planetary data collection, mission planning, resource allocation, or other related fields "
#     ),
# )

In [61]:
run_result_with_tools_azure = await Runner.run(
    starting_agent=upgraded_research_paper_assistant_azure,
    input=(
        "Get me information on rover navigation, planetary data collection, mission planning, resource allocation, or other related fields "
    ),
)

In [ ]:
# print(run_result_with_tools.raw_responses)

In [62]:
print(run_result_with_tools_azure.raw_responses)

[ModelResponse(output=[ResponseFunctionToolCall(arguments='{"user_query":"rover navigation in planetary exploration","retrieval_mode":"hybrid","top_k":5}', call_id='call_rKZHNs4XNybR9HAowAD9pdxo', name='get_research_papers', type='function_call', id='fc_02c6f2023447f3be006a172a06e5188197a3bcfabe0ffe903e', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"user_query":"planetary data collection methodologies","retrieval_mode":"hybrid","top_k":5}', call_id='call_cxLdStIMktCUMCKG384Jo6xM', name='get_research_papers', type='function_call', id='fc_02c6f2023447f3be006a172a06e53c81978842231e95cf1553', namespace=None, status='completed'), ResponseFunctionToolCall(arguments='{"user_query":"mission planning and optimization for planetary exploration","retrieval_mode":"hybrid","top_k":5}', call_id='call_a4glM8oibdXfrxog9HSxObpV', name='get_research_papers', type='function_call', id='fc_02c6f2023447f3be006a172a06e6008197926d58b7f43074d8', namespace=None, status='completed')

## Agent as Tools (Ochestration)


Add an image of the flow of these agents

In [ ]:
# # Define specialized agents for different research retrieval tasks
# research_paper_agent = Agent(
#     name="research_paper_agent",
#     instructions="""
#         You specialize in retrieving and summarizing academic research papers.
#         Use the get_research_papers tool to find relevant literature based on the user's query.
#         Always cite sources using [1], [2], etc., and focus on summarizing key findings,
#         methodologies, and implications of the studies retrieved.
#     """,
#     handoff_description="A research retrieval specialist with access to academic papers and literature databases.",
#     tools=[get_research_papers],
# )

# research_conversation_agent = Agent(
#     name="research_conversation_agent",
#     instructions="""
#         You specialize in retrieving and summarizing past research discussions and analyses.
#         Use the get_past_research_conversations tool to surface relevant prior sessions
#         or summaries that relate to the user's current topic of inquiry.
#         Present these as context and examples of prior analytical reasoning.
#     """,
#     handoff_description="A research memory specialist with access to prior academic discussions and analyses.",
#     tools=[get_past_research_conversations],
# )


In [63]:
# Azure companion: specialized agents with Azure deployment
research_paper_agent_azure = Agent(
    name="research_paper_agent_azure",
    model=AZURE_OPENAI_MODEL,
    instructions="""
        You specialize in retrieving and summarizing academic research papers.
        Use the get_research_papers tool to find relevant literature based on the user's query.
        Always cite sources using [1], [2], etc., and focus on summarizing key findings,
        methodologies, and implications of the studies retrieved.
    """,
    handoff_description="A research retrieval specialist with access to academic papers and literature databases.",
    tools=[get_research_papers],
)

research_conversation_agent_azure = Agent(
    name="research_conversation_agent_azure",
    model=AZURE_OPENAI_MODEL,
    instructions="""
        You specialize in retrieving and summarizing past research discussions and analyses.
        Use the get_past_research_conversations tool to surface relevant prior sessions
        or summaries that relate to the user's current topic of inquiry.
        Present these as context and examples of prior analytical reasoning.
    """,
    handoff_description="A research memory specialist with access to prior academic discussions and analyses.",
    tools=[get_past_research_conversations],
)

In [ ]:
# # Create an orchestrator agent that can coordinate both research retrieval agents
# orchestrator_agent = Agent(
#     name="research_assistant_orchestrator",
#     instructions=(
#         "You are a Research Orchestrator Assistant responsible for coordinating information retrieval "
#         "across multiple specialized research tools.\n\n"
#         "Your role is to help users explore, analyze, and synthesize academic research efficiently.\n\n"
#         "IMPORTANT RULES:\n"
#         "1. ALWAYS use translate_to_research_papers when a query mentions research papers, studies, or findings.\n"
#         "2. ALWAYS use translate_to_research_conversations when a query mentions previous discussions, analyses, or summaries.\n"
#         "3. If a query requests BOTH new research and past discussions, use BOTH tools in sequence.\n"
#         "4. NEVER attempt to provide research summaries without using your tools.\n"
#         "5. Each tool provides complementary context — use all appropriate tools for a comprehensive academic response.\n\n"
#         "After retrieving relevant results, synthesize them into a cohesive summary:\n"
#         "- Clearly distinguish between newly retrieved research and recalled past discussions.\n"
#         "- Cite sources using [1], [2], etc.\n"
#         "- Identify key insights, trends, and research gaps.\n"
#         "- Maintain an academic and objective tone."
#     ),
#     tools=[
#         research_paper_agent.as_tool(
#             tool_name="translate_to_research_papers",
#             tool_description="Retrieve and summarize relevant academic research papers and literature findings.",
#         ),
#         research_conversation_agent.as_tool(
#             tool_name="translate_to_research_conversations",
#             tool_description="Retrieve and summarize past research discussions or analyses related to the topic.",
#         ),
#     ],
# )


In [64]:
# Azure companion orchestrator
orchestrator_agent_azure = Agent(
    name="research_assistant_orchestrator_azure",
    model=AZURE_OPENAI_MODEL,
    instructions=(
        "You are a Research Orchestrator Assistant responsible for coordinating information retrieval "
        "across multiple specialized research tools.\n\n"
        "Your role is to help users explore, analyze, and synthesize academic research efficiently.\n\n"
        "IMPORTANT RULES:\n"
        "1. ALWAYS use translate_to_research_papers when a query mentions research papers, studies, or findings.\n"
        "2. ALWAYS use translate_to_research_conversations when a query mentions previous discussions, analyses, or summaries.\n"
        "3. If a query requests BOTH new research and past discussions, use BOTH tools in sequence.\n"
        "4. NEVER attempt to provide research summaries without using your tools.\n"
        "5. Each tool provides complementary context — use all appropriate tools for a comprehensive academic response.\n\n"
        "After retrieving relevant results, synthesize them into a cohesive summary:\n"
        "- Clearly distinguish between newly retrieved research and recalled past discussions.\n"
        "- Cite sources using [1], [2], etc.\n"
        "- Identify key insights, trends, and research gaps.\n"
        "- Maintain an academic and objective tone."
    ),
    tools=[
        research_paper_agent_azure.as_tool(
            tool_name="translate_to_research_papers",
            tool_description="Retrieve and summarize relevant academic research papers and literature findings.",
        ),
        research_conversation_agent_azure.as_tool(
            tool_name="translate_to_research_conversations",
            tool_description="Retrieve and summarize past research discussions or analyses related to the topic.",
        ),
    ],
)

In [ ]:
# # Final agent to synthesize information from all sources (Research use case)
# synthesizer_agent = Agent(
#     name="research_response_synthesizer",
#     instructions=(
#         "You create comprehensive, well-organized research summaries by combining information from multiple sources.\n\n"
#         "When organizing your response:\n"
#         "1) Start with a concise abstract-style overview (3–5 sentences) highlighting key findings and takeaways.\n"
#         "2) Clearly separate NEW LITERATURE FINDINGS from PAST RESEARCH DISCUSSIONS.\n"
#         "3) Cite sources using bracketed numbers [1], [2], etc., aligned with the retrieved items.\n"
#         "4) Emphasize methods, evidence strength, and limitations; avoid speculation beyond the provided context.\n"
#         "5) Use clear, scannable formatting (short paragraphs, bullet points where appropriate).\n"
#         "6) Conclude with open questions, gaps, or future work suggested by the literature.\n"
#         "7) If evidence is sparse, state this explicitly and avoid overgeneralization.\n"
#         "Tone: academic, objective, and precise."
#     ),
# )


In [65]:
# Azure companion synthesizer
synthesizer_agent_azure = Agent(
    name="research_response_synthesizer_azure",
    model=AZURE_OPENAI_MODEL,
    instructions=(
        "You create comprehensive, well-organized research summaries by combining information from multiple sources.\n\n"
        "When organizing your response:\n"
        "1) Start with a concise abstract-style overview (3–5 sentences) highlighting key findings and takeaways.\n"
        "2) Clearly separate NEW LITERATURE FINDINGS from PAST RESEARCH DISCUSSIONS.\n"
        "3) Cite sources using bracketed numbers [1], [2], etc., aligned with the retrieved items.\n"
        "4) Emphasize methods, evidence strength, and limitations; avoid speculation beyond the provided context.\n"
        "5) Use clear, scannable formatting (short paragraphs, bullet points where appropriate).\n"
        "6) Conclude with open questions, gaps, or future work suggested by the literature.\n"
        "7) If evidence is sparse, state this explicitly and avoid overgeneralization.\n"
        "Tone: academic, objective, and precise."
    ),
)

In [66]:
from agents import ItemHelpers, MessageOutputItem, trace
from agents import Runner  # assuming Runner is imported elsewhere; include here for clarity


In [ ]:
# async def research_assistant_workflow(user_query: str):
#     """Run the complete research assistant workflow (orchestrate retrieval + synthesize)."""
#     # 1) Have the research orchestrator decide which tools to invoke
#     with trace("Research Orchestrator"):
#         orchestrator_result = await Runner.run(orchestrator_agent, user_query)

#         # Debug/transparency: print intermediate orchestration steps
#         print("\n--- Research Orchestration Steps ---")
#         for item in orchestrator_result.new_items:
#             if isinstance(item, MessageOutputItem):
#                 text = ItemHelpers.text_message_output(item)
#                 if text:
#                     print(f"  - Retrieval step: {text}")

#         # 2) Synthesize all gathered information into a cohesive research summary
#         synthesizer_result = await Runner.run(
#             synthesizer_agent, orchestrator_result.to_input_list()
#         )

#         print(f"\n\n--- Final Research Synthesis ---\n{synthesizer_result.final_output}\n")

#     return synthesizer_result.final_output


In [67]:
async def research_assistant_workflow_azure(user_query: str):
    """Run the complete Azure research assistant workflow (orchestrate retrieval + synthesize)."""
    with trace("Research Orchestrator Azure"):
        orchestrator_result = await Runner.run(orchestrator_agent_azure, user_query)

        print("\n--- Research Orchestration Steps (Azure) ---")
        for item in orchestrator_result.new_items:
            if isinstance(item, MessageOutputItem):
                text = ItemHelpers.text_message_output(item)
                if text:
                    print(f"  - Retrieval step: {text}")

        synthesizer_result = await Runner.run(
            synthesizer_agent_azure, orchestrator_result.to_input_list()
        )

        print(f"\n\n--- Final Research Synthesis (Azure) ---\n{synthesizer_result.final_output}\n")

    return synthesizer_result.final_output

In [68]:
import asyncio
import nest_asyncio

# Apply nest_asyncio to patch the event loop
nest_asyncio.apply()

In [ ]:
# def run_virtual_research_assistant(query):
#     # Create a new event loop
#     loop = asyncio.new_event_loop()
#     asyncio.set_event_loop(loop)

#     # Run the async function and get the result
#     result = loop.run_until_complete(research_assistant_workflow(query))

#     # Clean up
#     loop.close()

#     return result

In [69]:
def run_virtual_research_assistant_azure(query):
    # Create a new event loop
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    # Run the async function and get the result
    result = loop.run_until_complete(research_assistant_workflow_azure(query))

    # Clean up
    loop.close()

    return result

In [ ]:
# # Now call the function this way
# query = input("What research topic can I help you with today? ")
# run_virtual_research_assistant(query)

In [70]:
# mission planning

# Azure companion invocation
query_azure = input("[Azure] What research topic can I help you with today? ")
run_virtual_research_assistant_azure(query_azure)


--- Research Orchestration Steps (Azure) ---
  - Retrieval step: ### Comprehensive Overview: Mission Planning

#### 1. **Findings from Recent Research Papers**
The retrieved studies emphasize specialized frameworks and methodologies for mission planning across diverse domains, particularly space science. Key highlights include:

- **Precise Time Synchronization in Satellite Missions:** Improvements in modeling time-transfer problems incorporating Earth's gravitational effects [1].
- **Enhanced Atmospheric Escape Models:** Development of kinetic simulations to refine models of atmospheric loss, offering benefits for missions to planetary bodies with evolving atmospheres [2].
- **Refined Planetary Detection Techniques:** Methods improving coronagraph sensitivity for direct planet imaging in interstellar missions [3].
- **Understanding Magnetohydrodynamics in Exploration Contexts:** Applications of MRI research to planetary core dynamics modeling [4].
- **Advanced Search Algorithms for P

"### Abstract Overview: Mission Planning Research\n\nMission planning encompasses frameworks, algorithms, and methodologies tailored for strategic and operational success in various domains, including space exploration, planetary science, and geophysical studies. Recent research advances include time synchronization techniques for satellite missions, atmospheric modeling for planetary exploration, and detection algorithms for circumbinary planets. Systematic improvements in computational approaches, observational sensitivity, and theoretical models are enabling more precise mission execution while addressing inherent scientific challenges.\n\n---\n\n### Recent Literature Findings\n\n#### 1. **Time Synchronization for Space Missions**\n- A study [1] proposes advanced equations for precise two-way time-transfer between satellites and ground stations, accounting for Earth's gravitational effects beyond basic models.\n- Application: Important for missions requiring accurate timing for sate

## Agentic Chat System


In [71]:
import datetime
import uuid

# Create chat_history table in Oracle
with conn.cursor() as cur:
    # Drop table if exists (for development)
    cur.execute("""
        BEGIN
            EXECUTE IMMEDIATE 'DROP TABLE chat_history';
        EXCEPTION WHEN OTHERS THEN
            IF SQLCODE != -942 THEN RAISE; END IF;
        END;
    """)
    
    # Create chat_history table
    cur.execute("""
        CREATE TABLE chat_history (
            id VARCHAR2(100) PRIMARY KEY,
            thread_id VARCHAR2(100) NOT NULL,
            role VARCHAR2(20) NOT NULL,
            message CLOB NOT NULL,
            timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        TABLESPACE USERS
    """)
    
    # Create index on thread_id and timestamp for efficient retrieval
    cur.execute("""
        CREATE INDEX idx_thread_timestamp 
        ON chat_history(thread_id, timestamp)
        TABLESPACE USERS
    """)
    
    conn.commit()
    print("✅ Table chat_history created successfully with index.")

✅ Table chat_history created successfully with index.


In [ ]:
# async def research_assistant_chat(user_query, thread_id=None):
#     """
#     Run the complete research assistant workflow with conversation history.
#     For each conversation turn:
#       - Stores the user's input and the assistant's output in Oracle along with a timestamp and thread_id.
#       - Retrieves and appends previous conversation history (ordered by timestamp) to the agent's input.
    
#     If no thread_id is provided, a new conversation session is started.
    
#     Returns:
#       tuple: (final_output, thread_id) where thread_id is the session identifier.
#     """
#     # Generate a new thread id if not provided
#     if thread_id is None:
#         thread_id = str(uuid.uuid4())
#         print(f"📝 New research conversation started with thread ID: {thread_id}")
#     else:
#         print(f"📝 Continuing research conversation with thread ID: {thread_id}")
    
#     # --- Step 1: Store the new user query in Oracle ---
#     message_id = str(uuid.uuid4())
    
#     with conn.cursor() as cur:
#         cur.execute("""
#             INSERT INTO chat_history (id, thread_id, role, message, timestamp)
#             VALUES (:id, :thread_id, :role, :message, CURRENT_TIMESTAMP)
#         """, {
#             'id': message_id,
#             'thread_id': thread_id,
#             'role': 'user',
#             'message': user_query
#         })
#         conn.commit()
    
#     # --- Step 2: Retrieve full conversation history for context ---
#     with conn.cursor() as cur:
#         cur.execute("""
#             SELECT role, message, timestamp
#             FROM chat_history
#             WHERE thread_id = :thread_id
#             ORDER BY timestamp ASC
#         """, {'thread_id': thread_id})
        
#         chat_history = cur.fetchall()
    
#     conversation_context = ""
#     for entry in chat_history:
#         role, message, timestamp = entry
#         if role == "user":
#             conversation_context += f"User: {message}\n"
#         else:
#             conversation_context += f"Assistant: {message}\n"
    
#     # --- Step 3: Run the orchestrator agent with the conversation context ---
#     with trace("Research Orchestrator"):
#         orchestrator_result = await Runner.run(orchestrator_agent, conversation_context)
    
#     # Print intermediate processing steps for debugging/transparency
#     print("\n--- Research Orchestrator Processing Steps ---")
#     for item in orchestrator_result.new_items:
#         if isinstance(item, MessageOutputItem):
#             text = ItemHelpers.text_message_output(item)
#             if text:
#                 print(f"  - Information gathering step: {text}")
    
#     # --- Step 4: Run the synthesizer agent to produce a cohesive response ---
#     synthesizer_result = await Runner.run(
#         synthesizer_agent, orchestrator_result.to_input_list()
#     )
    
#     # --- Step 5: Store the assistant's final output in Oracle ---
#     response_id = str(uuid.uuid4())
    
#     with conn.cursor() as cur:
#         cur.execute("""
#             INSERT INTO chat_history (id, thread_id, role, message, timestamp)
#             VALUES (:id, :thread_id, :role, :message, CURRENT_TIMESTAMP)
#         """, {
#             'id': response_id,
#             'thread_id': thread_id,
#             'role': 'assistant',
#             'message': synthesizer_result.final_output
#         })
#         conn.commit()
    
#     print(f"\n\n--- Final Research Response ---\n{synthesizer_result.final_output}\n")
    
#     return synthesizer_result.final_output, thread_id

In [73]:
async def research_assistant_chat_azure(user_query, thread_id=None):
    """Azure companion for research_assistant_chat using Azure orchestrator/synthesizer agents."""
    if thread_id is None:
        thread_id = str(uuid.uuid4())
        print(f"📝 New research conversation started with thread ID: {thread_id}")
    else:
        print(f"📝 Continuing research conversation with thread ID: {thread_id}")

    message_id = str(uuid.uuid4())
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO chat_history (id, thread_id, role, message, timestamp)
            VALUES (:id, :thread_id, :role, :message, CURRENT_TIMESTAMP)
        """, {
            'id': message_id,
            'thread_id': thread_id,
            'role': 'user',
            'message': user_query
        })
        conn.commit()

    with conn.cursor() as cur:
        cur.execute("""
            SELECT role, message, timestamp
            FROM chat_history
            WHERE thread_id = :thread_id
            ORDER BY timestamp ASC
        """, {'thread_id': thread_id})
        chat_history = cur.fetchall()

    conversation_context = ""
    for entry in chat_history:
        role, message, timestamp = entry
        if role == "user":
            conversation_context += f"User: {message}\n"
        else:
            conversation_context += f"Assistant: {message}\n"

    with trace("Research Orchestrator Azure"):
        orchestrator_result = await Runner.run(orchestrator_agent_azure, conversation_context)

    print("\n--- Research Orchestrator Processing Steps (Azure) ---")
    for item in orchestrator_result.new_items:
        if isinstance(item, MessageOutputItem):
            text = ItemHelpers.text_message_output(item)
            if text:
                print(f"  - Information gathering step: {text}")

    synthesizer_result = await Runner.run(
        synthesizer_agent_azure, orchestrator_result.to_input_list()
    )

    response_id = str(uuid.uuid4())
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO chat_history (id, thread_id, role, message, timestamp)
            VALUES (:id, :thread_id, :role, :message, CURRENT_TIMESTAMP)
        """, {
            'id': response_id,
            'thread_id': thread_id,
            'role': 'assistant',
            'message': synthesizer_result.final_output
        })
        conn.commit()

    print(f"\n\n--- Final Research Response (Azure) ---\n{synthesizer_result.final_output}\n")
    return synthesizer_result.final_output, thread_id

In [ ]:
# def run_research_assistant_chat(query, thread_id=None):
#     """
#     Run the research assistant synchronously.
#     Optionally, a thread_id can be provided to continue an existing conversation.
#     Returns a tuple (final_output, thread_id).
#     """
#     # Create a new event loop
#     loop = asyncio.new_event_loop()
#     asyncio.set_event_loop(loop)
    
#     # Run the async function and get the result
#     result, thread_id = loop.run_until_complete(
#         research_assistant_chat(query, thread_id=thread_id)
#     )
    
#     # Clean up the loop
#     loop.close()
    
#     return result, thread_id

In [74]:
def run_research_assistant_chat_azure(query, thread_id=None):
    """Run the Azure research assistant synchronously."""
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

    result, thread_id = loop.run_until_complete(
        research_assistant_chat_azure(query, thread_id=thread_id)
    )

    loop.close()
    return result, thread_id

In [ ]:
# def research_chat_session():
#     """
#     Launches a research chat session that continues until the user enters 'q', 'exit', or 'quit'.
#     The session uses a persistent thread_id to preserve conversation history.
#     """
#     print("🔬 Starting Research Paper Assistant Chat")
#     print("Type 'q', 'exit' or 'quit' to exit.\n")
    
#     session_thread_id = None
    
#     while True:
#         query = input("What research topic can I help you with today? ")
        
#         if query.lower() in ["q", "exit", "quit"]:
#             print("Exiting research chat session.")
#             break
        
#         response, session_thread_id = run_research_assistant_chat(
#             query, thread_id=session_thread_id
#         )
        
#         print(f"\n📚 Assistant: {response}\n")

In [75]:
def research_chat_session_azure():
    """Azure companion chat session with persistent thread memory."""
    print("🔬 Starting Research Paper Assistant Chat (Azure)")
    print("Type 'q', 'exit' or 'quit' to exit.\n")

    session_thread_id = None

    while True:
        query = input("[Azure] What research topic can I help you with today? ")

        if query.lower() in ["q", "exit", "quit"]:
            print("Exiting research chat session.")
            break

        response, session_thread_id = run_research_assistant_chat_azure(
            query, thread_id=session_thread_id
        )

        print(f"\n📚 Assistant: {response}\n")

In [ ]:
# # Start the research chat session
# research_chat_session()

In [76]:
# Start the Azure research chat session
research_chat_session_azure()

🔬 Starting Research Paper Assistant Chat (Azure)
Type 'q', 'exit' or 'quit' to exit.

📝 New research conversation started with thread ID: ba3e4aa0-6d9e-472a-a4d8-9ca7db8ff75b

--- Research Orchestrator Processing Steps (Azure) ---
  - Information gathering step: ### Comprehensive Summary: Mission Planning Research and Past Discussions

#### Findings from Academic Research
- **Mission Objective and Methodology**: One retrieved study focused on mission planning for the EPOXI mission targeting comet **103P/Hartley 2**. It aimed to confirm the comet's structural integrity with precise astrometry near aphelion, delivering key data on its **brightness and nucleus size**. Observations showed minimal activity and suggested a nucleus radius of ≤1 km, aiding trajectory and imaging strategies [1].
- **Implications**: The research highlighted the critical role of deep imaging and precise astrometric observations in preparing for space missions, particularly for smaller celestial objects near aphel

IntegrityError: ORA-01400: cannot insert NULL into ("SYSTEM"."CHAT_HISTORY"."MESSAGE")
Help: https://docs.oracle.com/error-help/db/ora-01400/

## Session Memory with Oracle AI Database


In [77]:
from typing import List, Optional, Union
from datetime import datetime
import oracledb
import json
import uuid

class OracleSession:
    """Custom Oracle session implementation following the Session protocol"""
    
    def __init__(
        self, 
        session_id: str, 
        connection,
        table_name: str = "chat_history"
    ):
        """
        Initialize Oracle session storage.
        
        Args:
            session_id: Unique identifier for this conversation session
            connection: Active oracledb connection object
            table_name: Name of the Oracle table storing session data
        """
        self.session_id = session_id
        self.conn = connection
        self.table_name = table_name
    
    async def get_items(self, limit: Optional[int] = None) -> List[dict]:
        """Retrieve conversation history for this session"""
        try:
            with self.conn.cursor() as cur:
                if limit:
                    cur.execute(f"""
                        SELECT message
                        FROM {self.table_name}
                        WHERE thread_id = :session_id
                        ORDER BY timestamp ASC
                        FETCH FIRST :limit ROWS ONLY
                    """, {'session_id': self.session_id, 'limit': limit})
                else:
                    cur.execute(f"""
                        SELECT message
                        FROM {self.table_name}
                        WHERE thread_id = :session_id
                        ORDER BY timestamp ASC
                    """, {'session_id': self.session_id})
                
                rows = cur.fetchall()
                
                items = []
                for row in rows:
                    # Deserialize JSON from CLOB
                    message_clob = row[0]
                    if message_clob:
                        message_str = message_clob.read() if hasattr(message_clob, 'read') else str(message_clob)
                        items.append(json.loads(message_str))
                
                return items
        
        except Exception as e:
            print(f"Error retrieving items: {e}")
            return []
    
    async def add_items(self, items: List[dict]) -> None:
        """Store new items for this session"""
        try:
            with self.conn.cursor() as cur:
                for item in items:
                    item_id = str(uuid.uuid4())
                    
                    # Serialize the entire item as JSON
                    message_json = json.dumps(item)
                    
                    # Extract role if available, otherwise default to 'system'
                    role = item.get('role', 'system')
                    
                    cur.execute(f"""
                        INSERT INTO {self.table_name} (id, thread_id, role, message, timestamp)
                        VALUES (:id, :session_id, :role, :message, CURRENT_TIMESTAMP)
                    """, {
                        'id': item_id,
                        'session_id': self.session_id,
                        'role': role,
                        'message': message_json
                    })
                
                self.conn.commit()
        
        except Exception as e:
            print(f"Error adding items: {e}")
            self.conn.rollback()
    
    async def pop_item(self, limit: Optional[int] = None) -> Optional[Union[dict, List[dict]]]:
        """
        Remove and return the most recent item(s) for this session.
        """
        try:
            with self.conn.cursor() as cur:
                # Pop a single most-recent item
                if not limit or limit <= 1:
                    cur.execute(f"""
                        SELECT id, message
                        FROM {self.table_name}
                        WHERE thread_id = :session_id
                        ORDER BY timestamp DESC
                        FETCH FIRST 1 ROW ONLY
                    """, {'session_id': self.session_id})
                    
                    row = cur.fetchone()
                    
                    if row:
                        item_id, message_clob = row
                        message_str = message_clob.read() if hasattr(message_clob, 'read') else str(message_clob)
                        item = json.loads(message_str)
                        
                        # Delete the item
                        cur.execute(f"""
                            DELETE FROM {self.table_name}
                            WHERE id = :id
                        """, {'id': item_id})
                        
                        self.conn.commit()
                        return item
                    
                    return None
                
                # Pop multiple most-recent items
                cur.execute(f"""
                    SELECT id, message
                    FROM {self.table_name}
                    WHERE thread_id = :session_id
                    ORDER BY timestamp DESC
                    FETCH FIRST :limit ROWS ONLY
                """, {'session_id': self.session_id, 'limit': limit})
                
                rows = cur.fetchall()
                
                if not rows:
                    return []
                
                items = []
                ids_to_delete = []
                
                for row in rows:
                    item_id, message_clob = row
                    message_str = message_clob.read() if hasattr(message_clob, 'read') else str(message_clob)
                    items.append(json.loads(message_str))
                    ids_to_delete.append(item_id)
                
                # Delete all items
                for item_id in ids_to_delete:
                    cur.execute(f"""
                        DELETE FROM {self.table_name}
                        WHERE id = :id
                    """, {'id': item_id})
                
                self.conn.commit()
                return items
        
        except Exception as e:
            print(f"Error popping item(s): {e}")
            self.conn.rollback()
            return None if (not limit or limit <= 1) else []
    
    async def clear_session(self) -> None:
        """Clear all items for this session"""
        try:
            with self.conn.cursor() as cur:
                cur.execute(f"""
                    DELETE FROM {self.table_name}
                    WHERE thread_id = :session_id
                """, {'session_id': self.session_id})
                
                self.conn.commit()
                print(f"✅ Session {self.session_id} cleared successfully.")
        
        except Exception as e:
            print(f"Error clearing session: {e}")
            self.conn.rollback()
    
    def close(self) -> None:
        """
        Note: Connection is managed externally, so we don't close it here.
        """
        pass

Basic Example of an Agent with Session Memory


In [ ]:
# # Create an agent
# research_agent = Agent(
#     name="Assistant",
#     instructions="Research the topic and return the most relevant information.",
# )

In [78]:
# Azure companion agent with explicit Azure deployment model
research_agent_azure = Agent(
    name="Assistant (Azure)",
    model=AZURE_OPENAI_MODEL,
    instructions="Research the topic and return the most relevant information.",
)

In [ ]:
# # Create an Oracle session instance
# session = OracleSession(
#     session_id="conversation_123", 
#     connection=conn,
#     table_name="chat_history"
# )

In [79]:
# Azure companion Oracle session instance
session_azure = OracleSession(
    session_id="conversation_azure_123",
    connection=conn,
    table_name="chat_history"
)

In [ ]:
# # First turn
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="Hi my name is Richmond, and I am a AI Memory Engineer researching LLMs and Agent Memory",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [80]:
# First turn (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Hi my name is Richmond, and I am a AI Memory Engineer researching LLMs and Agent Memory",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): Hi Richmond! That sounds like a fascinating field of research. Working on AI memory systems, particularly with Large Language Models (LLMs) and agent memory, is a critical area shaping the future of AI. Here's a curated overview of concepts, challenges, and emerging ideas that might align with your work:

---

### **Key Topics in LLMs and Agent Memory**

#### 1. **Memory Architectures in LLMs**
   - LLMs like GPT-4 have limitations regarding memory persistence. They rely on context windows (e.g., 8k, 32k tokens) rather than episodic or long-term memory.
   - Building **long-term memory systems** for LLMs is an active area of exploration using techniques like:
     - **External memory modules** (vector databases for information retrieval).
     - **Recurrent memory patterns** to store knowledge persistently across sessions.
     - Integration with graph-based systems to connect historical inputs.

#### 2. **Agent Memory Models**
   Independent of pure LLM designs, age

In [ ]:
# # Second turn
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="What is a paper that introduces the attention mechanism in LLMs?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [81]:
# Second turn (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="What is a paper that introduces the attention mechanism in LLMs?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): The foundational paper that introduced the **attention mechanism**, which revolutionized natural language processing and ultimately led to the development of Large Language Models (LLMs), is:

### **"Attention Is All You Need" (Vaswani et al., 2017)**  
- **Authors**: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Lukasz Kaiser, and Illia Polosukhin.
- **Published in**: NeurIPS 2017 (Conference on Neural Information Processing Systems).
 
You can find the paper here: [Link to PDF](https://arxiv.org/abs/1706.03762)

---

### **Key Contributions of the Paper**

1. **Introduction of the Transformer Architecture**:
   - The paper proposed a new neural network architecture called the "Transformer," which relies entirely on attention mechanisms, replacing recurrent and convolutional networks traditionally used for sequence-based tasks.

2. **Self-Attention Mechanism**:
   - Attention mechanisms allow the model to focus on relevant 

In [ ]:
# # Third turn, the agent will remember the previous conversation
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="Who were the authors of the paper?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [82]:
# Third turn, the agent will remember the previous conversation (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Who were the authors of the paper?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): The authors of the seminal paper **"Attention Is All You Need"**, published in 2017, are as follows:

1. **Ashish Vaswani**  
2. **Noam Shazeer**  
3. **Niki Parmar**  
4. **Jakob Uszkoreit**  
5. **Llion Jones**  
6. **Aidan N. Gomez**  
7. **Łukasz Kaiser**  
8. **Illia Polosukhin**  

These researchers were primarily affiliated with **Google Research**, except for Illia Polosukhin, who later co-founded **NEAR Protocol**. The paper was presented at the **NeurIPS 2017 (Conference on Neural Information Processing Systems)** and has since become one of the most influential papers in machine learning and natural language processing. 

Let me know if you'd like further details on their contributions or insights!


In [ ]:
# # Fourth turn - continuing the conversation
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="What was the year of publication?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [83]:
# Fourth turn - continuing the conversation (Azure companion)
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="What was the year of publication?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): The paper **"Attention Is All You Need"** was published in **2017**. It was presented at the **31st Annual Conference on Neural Information Processing Systems (NeurIPS 2017)**, which took place in **Long Beach, California**, from December 4 to 9, 2017. 

This publication has since become one of the most cited and influential works in the field of machine learning, marking the birth of the **Transformer architecture**, which underpins modern Large Language Models (LLMs).


In [ ]:
# # Change the conversation subject and ensure the agent does't remember the previous conversation
# # Specifiying pop without limit will remove the last item in the session
# await session.pop_item(limit=7)

In [84]:
# Change the conversation subject for Azure companion session
await session_azure.pop_item(limit=7)

[{'id': 'msg_060c6360dad56b20006a172c01f0d08194b377f3ea414957fa',
  'content': [{'annotations': [],
    'text': 'The paper **"Attention Is All You Need"** was published in **2017**. It was presented at the **31st Annual Conference on Neural Information Processing Systems (NeurIPS 2017)**, which took place in **Long Beach, California**, from December 4 to 9, 2017. \n\nThis publication has since become one of the most cited and influential works in the field of machine learning, marking the birth of the **Transformer architecture**, which underpins modern Large Language Models (LLMs).',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'},
 {'content': 'What was the year of publication?', 'role': 'user'},
 {'id': 'msg_060c6360dad56b20006a172bf54a248194a2c2aaedee5d25f3',
  'content': [{'annotations': [],
    'text': 'The authors of the seminal paper **"Attention Is All You Need"**, published in 2017, are as follows:\n\n1. **

In [ ]:
# # Fifth turn: The agent should not remember the conversations about the paper
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="What paper have we been talking about?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [85]:
# Fifth turn: The Azure agent should not remember the conversations about the paper
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="What paper have we been talking about?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): Ah, Richmond, great to meet you! As your AI assistant, I currently do not have memory in this context, so I don't retain details about past conversations or ongoing topics. If you’ve referred to a particular research paper earlier, you’ll need to mention it again. 

However, since you're an AI Memory Engineer researching LLMs (Large Language Models) and Agent Memory, perhaps you're discussing a prominent recent paper on memory structures, retrieval-augmented generation, or agent behavior? Papers like **"Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks"** or OpenAI's publications on systems like GPT could be relevant.

If you share any specific details about the paper or its topic, I’d be happy to help explore it further with you!


In [ ]:
# # Because we limited the session to a few items, the agent should still remember our name at the introduction
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="Do you still remember my name?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [86]:
# Check whether Azure companion session still remembers the name
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Do you still remember my name?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): Yes, you mentioned your name is **Richmond**! While I don’t have long-term memory within this conversation, I can keep track of details (like your name) temporarily while we’re still actively chatting. Once our session ends, though, that information won't persist unless I gain memory retention capabilities in the future. Let me know how I can assist you! 😊


In [ ]:
# # Clear the session
# await session.clear_session()

In [87]:
# Clear the Azure companion session
await session_azure.clear_session()

✅ Session conversation_azure_123 cleared successfully.


In [ ]:
# # Because we limited the session to 3 items, the agent should still remember our name at the introduction
# result_from_research_agent = await Runner.run(
#     starting_agent=research_agent,
#     input="Do you still remember my name?",
#     session=session
# )

# print(f"Assistant: {result_from_research_agent.final_output}")

In [88]:
# After clearing, Azure companion session should not remember prior details
result_from_research_agent_azure = await Runner.run(
    starting_agent=research_agent_azure,
    input="Do you still remember my name?",
    session=session_azure
)

print(f"Assistant (Azure): {result_from_research_agent_azure.final_output}")

Assistant (Azure): I don’t have memory of past interactions, so I don’t know your name unless you tell me now! If you'd like, feel free to share it again.
